# Repository Maintenance Notebook

This notebook demonstrates techniques for rewriting Git history, trimming dependencies, and reorganizing the source tree programmatically using Python and Git. It is intended for educational and operational reference rather than execution in this environment.

In [ ]:
# Section 1: Import Required Libraries
# We'll bring in GitPython for git operations; os and shutil for filesystem work; subprocess for running shell commands.
import os
import shutil
import subprocess
import sys
from pathlib import Path

# GitPython may not be installed in this environment, but we import it assuming a normal dev setup.
try:
    import git
except ImportError:
    git = None  # will fallback to subprocess calls

print("Imports complete")

## Section 2: Access Git Repository

Open the current repository or clone a copy for safe operations. Inspect branches and commits to ensure we have a clean starting point.

In [ ]:
repo_path = Path.cwd()
print(f"Current working directory: {repo_path}")

if git:
    repo = git.Repo(repo_path)
    print(f"Active branch: {repo.active_branch}")
    print(f"Latest commit: {repo.head.commit.hexsha}")
else:
    print("GitPython not available; will use subprocess instead.")
    branch = subprocess.check_output(["git", "branch", "--show-current"]).strip().decode()
    commit = subprocess.check_output(["git", "rev-parse", "HEAD"]).strip().decode()
    print(f"Active branch: {branch}")
    print(f"Latest commit: {commit}")

## Section 3: Rewrite Git History

Use `git filter-repo` (or `git filter-branch` as fallback) to remove sensitive or bulky paths from history. In a script we can invoke the module directly.

In [ ]:
# Example function to run git-filter-repo with Python module call

def rewrite_history(paths_to_remove):
    cmd = [
        sys.executable, "-m", "git_filter_repo",
        "--force",
    ]
    for p in paths_to_remove:
        cmd.extend(["--invert-paths", "--path", p])
    print("Running history rewrite: ", " ".join(cmd))
    subprocess.run(cmd, check=True)

# sample call (commented out to avoid accidental execution)
# rewrite_history(["uncategorized_for_review", "galactica/databases"])
print("History rewrite function defined")

## Section 4: Trim Unused Dependencies

Parse `requirements.txt`, compare against actual imports in the codebase, and remove entries that aren't referenced by any module.

In [ ]:
def parse_imports(root):
    """Walk python files to gather top-level import names."""
    imports = set()
    for pyfile in Path(root).rglob("*.py"):
        with open(pyfile, 'r', encoding='utf-8', errors='ignore') as f:
            for line in f:
                line=line.strip()
                if line.startswith('import ') or line.startswith('from '):
                    parts=line.replace(',', ' ').split()
                    if parts[0]=='import':
                        imports.add(parts[1].split('.')[0])
                    elif parts[0]=='from' and parts[1] != '.':
                        imports.add(parts[1].split('.')[0])
    return imports

imports = parse_imports(repo_path)
print(f"Discovered imports: {sorted(list(imports))[:20]}...")

# read requirements
reqs = []
with open(repo_path / 'requirements.txt') as f:
    for line in f:
        pkg=line.strip().split('==')[0]
        reqs.append(pkg)

# naive trim: keep reqs that appear in imports
trimmed = [r for r in reqs if r in imports]
print(f"Trimmed requirements (~may need manual review): {trimmed[:20]}...")